In [ ]:
library(Seurat)
library(dplyr)
library(data.table)
library(ggplot2)
source("~/Projects/heads/clustering.r")

In [ ]:
data_dir = "/gpfs/gibbs/pi/braun/zy325"

In [ ]:
obj = readRDS(file.path(data_dir,"processed","theta1_dims50","scrcc_immune.rds"))

In [ ]:
obj = clustering(obj,
                plot_QC_metrics = F,
                group.by.vars = "batch_lab",
                harmony_theta=1, dims = 1:20)

In [ ]:
saveRDS(obj,file=file.path(data_dir,"processed","theta1_dims50","scrcc_immune_clustered.rds"))

In [ ]:
obj = readRDS(file.path(data_dir,"processed","theta1_dims50","scrcc_immune_clustered.rds"))

In [ ]:
options(repr.plot.width=8,repr.plot.height=10)
DimPlot(obj, label=T) + NoLegend()

In [ ]:
options(repr.plot.width=16,repr.plot.height=8)
DimPlot(obj,group.by = "batch_lab") | DimPlot(obj,group.by = "batch_seq_rna") #+ NoLegend()

In [ ]:
VlnPlot(obj,features = c("nCount_RNA","nFeature_RNA","percent.mt"),pt.size = 0,stack = T,flip = T)

In [ ]:
options(repr.plot.width=15,repr.plot.height=10)
VlnPlot(obj,features = c("CD3D","CD3E","CD79A","MZB1","NCAM1","FCGR3A",
    "CD68","S100A8","CD14","CD163","C1QC","LYZ","CLEC9A","CD1C","CPA3","LILRA4",
    "HBB","PPBP","EPCAM","ALDOB","PLVAP","COL1A1","PTPRC"),pt.size = 0,stack = T,flip = T)

In [ ]:
options(repr.plot.width=15,repr.plot.height=10)
obj = NormalizeData(obj,assay = "AIR")
VlnPlot(obj,features = c("TRBC1","TRAC","TRDC","TRDV2","TRGV9","TRAV1-2","TRAV11"),assay = "AIR",pt.size = 0,stack = T,flip = T)

In [ ]:
### TCR mapping ###
tcrs = list.dirs("/gpfs/gibbs/project/braun/zy325/scrcc/raw/tcr_airrflow_output/cellranger",recursive = F)
tcrs = lapply(file.path(tcrs,"outs","filtered_contig_annotations.csv"),function(x){
    cr = fread(x) %>% mutate(
        sample_id2=gsub("^.*cellranger\\/","",gsub("\\/outs.*$","",x)),
        sample_barcode=paste0(sample_id2,'_',barcode))
    return(cr)
}) %>% rbindlist

obj$barcode = gsub("^.*removed_","",obj$name)
obj$sample_barcode = paste0(obj$sample_id2,"_",obj$barcode)
obj$wTCR = obj$sample_barcode %in% tcrs$sample_barcode

options(repr.plot.width=8,repr.plot.height=7)
DimPlot(obj,group.by = "wTCR") #+ NoLegend()

In [ ]:
############ scrcc_immune_clustered.rds ############

# Find CD3D/CD79A/CD56+ clusters
obj$CD3D_data = obj@assays$RNA@layers$data[which(rownames(obj)=="CD3D"),]
obj$CD79A_data = obj@assays$RNA@layers$data[which(rownames(obj)=="CD79A"),]
obj$CD56_data = obj@assays$RNA@layers$data[which(rownames(obj)=="NCAM1"),]

lymphoid_clusters = obj@meta.data %>% group_by(seurat_clusters) %>% 
    summarise(
        CD3D_data_q3 = quantile(CD3D_data,probs = .75),
        CD79A_data_q3 = quantile(CD79A_data,probs = .75),
        CD56_data_q3 = quantile(CD56_data,probs = .75), .groups="drop") %>% 
    filter(CD3D_data_q3>0 | CD79A_data_q3>0 | CD56_data_q3>0) %>% .$seurat_clusters %>% as.character

# Check ILC clusters
# 2 - TOX+ THEMIS+ 
lymphoid_clusters = c(lymphoid_clusters,2)

# Check contamination
# High T & Myeloid: 9 # T-Mph doublets, into myeloid object
# High hypoxic genes (CA9+ EPO+ NDUFA4L2): 10
# Small clusters: 27, 28
lymphoid_clusters = setdiff(lymphoid_clusters,c(9,10,27,28))

In [ ]:
############ theta1_dims50 ############

# Find CD3D/CD79A/CD56+ clusters
obj$CD3D_data = obj@assays$RNA@layers$data[which(rownames(obj)=="CD3D"),]
obj$CD79A_data = obj@assays$RNA@layers$data[which(rownames(obj)=="CD79A"),]
obj$CD56_data = obj@assays$RNA@layers$data[which(rownames(obj)=="NCAM1"),]

lymphoid_clusters = obj@meta.data %>% group_by(seurat_clusters) %>% 
    summarise(
        CD3D_data_q3 = quantile(CD3D_data,probs = .75),
        CD79A_data_q3 = quantile(CD79A_data,probs = .75),
        CD56_data_q3 = quantile(CD56_data,probs = .75), .groups="drop") %>% 
    filter(CD3D_data_q3>0 | CD79A_data_q3>0 | CD56_data_q3>0) %>% .$seurat_clusters %>% as.character

# Check ILC clusters

# Check contamination
# High-mito: 27
# High T & Macro: 22

lymphoid_clusters = setdiff(lymphoid_clusters,c(22,27))

In [ ]:
### Doublet collection ###

dbls = readRDS(file.path(data_dir,"processed","theta1_dims50","dbl_list.rds"))
dbl = subset(obj, `RNA_snn_res.0.5` == 22)
dbl$anno_dbl = "Lym-Mye"
dbls[["immune"]] = dbl
saveRDS(dbls,file = file.path(data_dir,"processed","theta1_dims50","dbl_list.rds"))

In [ ]:
obj$lineage2 = case_when(
    obj$`RNA_snn_res.0.5` %in% lymphoid_clusters~"Lymphoid",
    obj$`RNA_snn_res.0.5` %in% c(22,27)~"contamination",
    TRUE~"Myeloid") 

In [ ]:
options(repr.plot.width=8,repr.plot.height=8)
DimPlot(obj,group.by = "lineage2") + NoLegend()

In [ ]:
library(ggsankey)
obj@meta.data %>% 
    filter(!is.na(anno_cd8t)) %>%
    make_long(lineage2,anno_cd8t) %>% # `RNA_snn_res.0.5`
    ggplot(aes(x = x, 
               next_x = next_x, 
               node = node, 
               next_node = next_node,
               fill = factor(node),
               label = node)) +
    geom_sankey(flow.alpha = 0.5, node.color = 1) +
    geom_sankey_label(size = 3, color = 1, fill = "white") +
    theme_sankey(base_size = 16) + NoLegend()

table(obj$lineage1[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing = T)

In [ ]:
obj@meta.data %>% 
    make_long(lineage2,anno2) %>% # `RNA_snn_res.0.5`
    ggplot(aes(x = x, 
               next_x = next_x, 
               node = node, 
               next_node = next_node,
               fill = factor(node),
               label = node)) +
    geom_sankey(flow.alpha = 0.5, node.color = 1) +
    geom_sankey_label(size = 3, color = 1, fill = "white") +
    theme_sankey(base_size = 16) + NoLegend()

In [ ]:
obj@meta.data %>% head

In [ ]:
obj$lineage2_clusters = obj$`RNA_snn_res.0.5`
obj$`RNA_snn_res.0.5` = NULL
obj$seurat_clusters = NULL

In [ ]:
saveRDS(subset(obj,lineage2=="Myeloid"),file=file.path(data_dir,"processed","theta1_dims50","scrcc_myeloid.rds"))

In [ ]:
m = FindMarkers(obj,`ident.1` = 22,only.pos = T,logfc.threshold = 1)
m %>% filter(p_val_adj<0.01) %>% arrange(desc(avg_log2FC)) %>% filter(abs(pct.1-pct.2)>.1)